In [6]:
import pandas as pd

# Paths to the batter CSVs for 2023, 2024, 2025
batters_2023_path = "/home/surya/AgentXI/data/2023-batters.csv"
batters_2024_path = "/home/surya/AgentXI/data/2024-batters.csv"
batters_2025_path = "/home/surya/AgentXI/data/2025-batters.csv"

# Load the CSV files
df_2023 = pd.read_csv(batters_2023_path)
df_2024 = pd.read_csv(batters_2024_path)
df_2025 = pd.read_csv(batters_2025_path)

# Add year column to each dataframe
df_2023['Year'] = 2023
df_2024['Year'] = 2024
df_2025['Year'] = 2025

# Merge all three dataframes on Player only (not Team)
combined = df_2025.merge(
    df_2024, on='Player', how='outer', suffixes=('_2025', '_2024')
).merge(
    df_2023, on='Player', how='outer'
)

# Rename 2023 columns (they don't have suffix yet)
for col in df_2023.columns:
    if col not in ['Player', 'Year']:
        if col in combined.columns and f'{col}_2025' in combined.columns:
            combined.rename(columns={col: f'{col}_2023'}, inplace=True)

# Combine teams from all years
def combine_teams(row):
    teams = []
    for year_suffix in ['_2025', '_2024', '_2023']:
        team_col = f'Team{year_suffix}'
        if team_col in row.index and pd.notna(row[team_col]):
            team = str(row[team_col])
            if team not in teams:
                teams.append(team)
    return ', '.join(teams) if teams else None

combined['combined_teams'] = combined.apply(combine_teams, axis=1)

# Get all metric columns (excluding Player, Team, Year, Rank)
metric_cols = [col for col in df_2025.columns if col not in ['Player', 'Team', 'Year', 'Rank']]

# Calculate weighted batting impact per innings
def calculate_weighted_impact(row):
    val_2025 = row.get('Batting impact/innings_2025', None)
    val_2024 = row.get('Batting impact/innings_2024', None)
    val_2023 = row.get('Batting impact/innings_2023', None)
    
    # Convert to numeric, handling NaN
    vals = []
    for v in [val_2025, val_2024, val_2023]:
        if pd.notna(v):
            try:
                vals.append(float(v))
            except (ValueError, TypeError):
                vals.append(None)
        else:
            vals.append(None)
    
    val_2025, val_2024, val_2023 = vals
    
    # Count how many years have data
    years_present = sum(v is not None for v in [val_2025, val_2024, val_2023])
    
    if years_present == 3:
        # All three years: 0.50*2025 + 0.35*2024 + 0.15*2023
        return 0.50 * val_2025 + 0.35 * val_2024 + 0.15 * val_2023, years_present
    elif years_present == 2:
        # Two years: 0.6*latest + 0.4*second_latest
        if val_2025 is not None and val_2024 is not None:
            return 0.6 * val_2025 + 0.4 * val_2024, years_present
        elif val_2025 is not None and val_2023 is not None:
            return 0.6 * val_2025 + 0.4 * val_2023, years_present
        elif val_2024 is not None and val_2023 is not None:
            return 0.6 * val_2024 + 0.4 * val_2023, years_present
    elif years_present == 1:
        # One year: 0.85*only_year
        if val_2025 is not None:
            return 0.85 * val_2025, years_present
        elif val_2024 is not None:
            return 0.85 * val_2024, years_present
        elif val_2023 is not None:
            return 0.85 * val_2023, years_present
    
    return None, 0

combined[['weighted_batting_impact_per_innings', 'active_years_count']] = combined.apply(
    lambda row: pd.Series(calculate_weighted_impact(row)), axis=1
)

# Create a new dataframe with only Player and combined columns
result = pd.DataFrame()
result['Player'] = combined['Player']
result['combined_teams'] = combined['combined_teams']

# Create combined columns with [2025, 2024, 2023] values
for col in metric_cols:
    col_2025 = f'{col}_2025' if f'{col}_2025' in combined.columns else col
    col_2024 = f'{col}_2024'
    col_2023 = f'{col}_2023'
    
    result[f'{col}_combined'] = combined.apply(
        lambda row: [
            row.get(col_2025, None),
            row.get(col_2024, None),
            row.get(col_2023, None)
        ],
        axis=1
    )

# Add weighted batting impact and active years count
result['weighted_batting_impact_per_innings'] = combined['weighted_batting_impact_per_innings']
result['active_years_count'] = combined['active_years_count']

# Sort by weighted batting impact per innings in descending order
result = result.sort_values('weighted_batting_impact_per_innings', ascending=False, na_position='last')

# Reset index
result = result.reset_index(drop=True)

# Save to a new CSV
combined_out_path = "/home/surya/AgentXI/data/combined-batters.csv"
result.to_csv(combined_out_path, index=False)
print(f"Combined batters stats written to {combined_out_path}")

Combined batters stats written to /home/surya/AgentXI/data/combined-batters.csv


In [7]:
import pandas as pd

# Paths to the bowler CSVs for 2023, 2024, 2025
bowlers_2023_path = "/home/surya/AgentXI/data/2023-bowlers.csv"
bowlers_2024_path = "/home/surya/AgentXI/data/2024-bowlers.csv"
bowlers_2025_path = "/home/surya/AgentXI/data/2025-bowlers.csv"

# Load the CSV files
df_2023 = pd.read_csv(bowlers_2023_path)
df_2024 = pd.read_csv(bowlers_2024_path)
df_2025 = pd.read_csv(bowlers_2025_path)

# Add year column to each dataframe
df_2023['Year'] = 2023
df_2024['Year'] = 2024
df_2025['Year'] = 2025

# Merge all three dataframes on Player only (not Team)
combined = df_2025.merge(
    df_2024, on='Player', how='outer', suffixes=('_2025', '_2024')
).merge(
    df_2023, on='Player', how='outer'
)

# Rename 2023 columns (they don't have suffix yet)
for col in df_2023.columns:
    if col not in ['Player', 'Year']:
        if col in combined.columns and f'{col}_2025' in combined.columns:
            combined.rename(columns={col: f'{col}_2023'}, inplace=True)

# Combine teams from all years
def combine_teams(row):
    teams = []
    for year_suffix in ['_2025', '_2024', '_2023']:
        team_col = f'Team{year_suffix}'
        if team_col in row.index and pd.notna(row[team_col]):
            team = str(row[team_col])
            if team not in teams:
                teams.append(team)
    return ', '.join(teams) if teams else None

combined['combined_teams'] = combined.apply(combine_teams, axis=1)

# Get all metric columns (excluding Player, Team, Year, Rank)
metric_cols = [col for col in df_2025.columns if col not in ['Player', 'Team', 'Year', 'Rank']]

# Calculate weighted bowling impact per match
def calculate_weighted_impact(row):
    val_2025 = row.get('Bowling impact/mat_2025', None)
    val_2024 = row.get('Bowling impact/mat_2024', None)
    val_2023 = row.get('Bowling impact/mat_2023', None)
    
    # Convert to numeric, handling NaN
    vals = []
    for v in [val_2025, val_2024, val_2023]:
        if pd.notna(v):
            try:
                vals.append(float(v))
            except (ValueError, TypeError):
                vals.append(None)
        else:
            vals.append(None)
    
    val_2025, val_2024, val_2023 = vals
    
    # Count how many years have data
    years_present = sum(v is not None for v in [val_2025, val_2024, val_2023])
    
    if years_present == 3:
        # All three years: 0.50*2025 + 0.35*2024 + 0.15*2023
        return 0.50 * val_2025 + 0.35 * val_2024 + 0.15 * val_2023, years_present
    elif years_present == 2:
        # Two years: 0.6*latest + 0.4*second_latest
        if val_2025 is not None and val_2024 is not None:
            return 0.6 * val_2025 + 0.4 * val_2024, years_present
        elif val_2025 is not None and val_2023 is not None:
            return 0.6 * val_2025 + 0.4 * val_2023, years_present
        elif val_2024 is not None and val_2023 is not None:
            return 0.6 * val_2024 + 0.4 * val_2023, years_present
    elif years_present == 1:
        # One year: 0.85*only_year
        if val_2025 is not None:
            return 0.85 * val_2025, years_present
        elif val_2024 is not None:
            return 0.85 * val_2024, years_present
        elif val_2023 is not None:
            return 0.85 * val_2023, years_present
    
    return None, 0

combined[['weighted_bowling_impact_per_match', 'active_years_count']] = combined.apply(
    lambda row: pd.Series(calculate_weighted_impact(row)), axis=1
)

# Create a new dataframe with only Player and combined columns
result = pd.DataFrame()
result['Player'] = combined['Player']
result['combined_teams'] = combined['combined_teams']

# Create combined columns with [2025, 2024, 2023] values
for col in metric_cols:
    col_2025 = f'{col}_2025' if f'{col}_2025' in combined.columns else col
    col_2024 = f'{col}_2024'
    col_2023 = f'{col}_2023'
    
    result[f'{col}_combined'] = combined.apply(
        lambda row: [
            row.get(col_2025, None),
            row.get(col_2024, None),
            row.get(col_2023, None)
        ],
        axis=1
    )

# Add weighted bowling impact and active years count
result['weighted_bowling_impact_per_match'] = combined['weighted_bowling_impact_per_match']
result['active_years_count'] = combined['active_years_count']

# Sort by weighted bowling impact per match in descending order
result = result.sort_values('weighted_bowling_impact_per_match', ascending=False, na_position='last')

# Reset index
result = result.reset_index(drop=True)

# Save to a new CSV
combined_out_path = "/home/surya/AgentXI/data/combined-bowlers.csv"
result.to_csv(combined_out_path, index=False)
print(f"Combined bowlers stats written to {combined_out_path}")

Combined bowlers stats written to /home/surya/AgentXI/data/combined-bowlers.csv


In [3]:
combined

,Rank,Player,Team,Batting Impact,Batting impact/innings,Batted Innings,Runs,Impact Runs
0,1,Shubman Gill,GT,1001.8,58.9,17,890,941.4
1,2,Faf du Plessis,RCB,828.1,59.1,14,730,790.1
2,3,Yashasvi Jaiswal,RR,725.1,51.7,14,625,676.1
3,4,Suryakumar Yadav,MI,701.3,43.8,16,605,648.8
4,5,Ruturaj Gaikwad,CSK,642.8,42.8,15,590,619.7
...,...,...,...,...,...,...,...,...
501,159,Donovan Ferreira,DC,-6.7,-6.7,1,1,-3.1
502,160,Wanindu Hasaranga,RR,-10.4,-2.1,5,9,-0.1
503,161,Kunal Singh Rathore,RR,-14.6,-14.6,1,0,-6.9
504,162,Moeen Ali,KKR,-14.9,-7.5,2,5,-2.2


In [21]:



import json 
import pandas as pd

# Load combined batters and bowlers data
combined_batters = pd.read_csv('/home/surya/AgentXI/data/new-combined-batters.csv')
combined_bowlers = pd.read_csv('/home/surya/AgentXI/data/new-combined-bowlers.csv')

# Get all player names from both dataframes
batters_players = set(combined_batters['Player'].tolist())
bowlers_players = set(combined_bowlers['Player'].tolist())
all_players = batters_players.union(bowlers_players)

with open('/home/surya/AgentXI/data/squads.json', 'r') as f:
    data = json.load(f)

# print(data)
missing_players_list = []
for team in data:
    print(f"Team: {team}")
    for player_type in data[team]:
        print(f"Player type: {player_type}")
        for player in data[team][player_type]:
            # print(player)
            # print(player["ShortName"])
            player_name = player["ShortName"]
            if player_name not in all_players:
                print(f"  ⚠️  Player not found in combined data: {player_name}")
                missing_players_list.append(player_name)
        print("--------------------------------")


Team: MI
Player type: BOWLER
--------------------------------
Player type: BATSMAN
--------------------------------
Player type: ALL ROUNDER
--------------------------------
Player type: WICKET KEEPER
--------------------------------
Team: RCB
Player type: BATSMAN
--------------------------------
Player type: BOWLER
--------------------------------
Player type: WICKET KEEPER
--------------------------------
Player type: ALL ROUNDER
--------------------------------
Team: RR
Player type: ALL ROUNDER
--------------------------------
Player type: BATSMAN
--------------------------------
Player type: BOWLER
--------------------------------
Player type: WICKET KEEPER
--------------------------------
Team: LSG
Player type: WICKET KEEPER
--------------------------------
Player type: ALL ROUNDER
--------------------------------
Player type: BATSMAN
--------------------------------
Player type: BOWLER
--------------------------------
Team: KKR
Player type: ALL ROUNDER
---------------------------

In [18]:
len(missing_players_list)

# Add missing players to combined_batters with NaN values
for player_name in missing_players_list:
    if player_name not in batters_players:
        new_row = {col: pd.NA for col in combined_batters.columns}
        new_row['Player'] = player_name
        combined_batters = pd.concat([combined_batters, pd.DataFrame([new_row])], ignore_index=True)

# Add missing players to combined_bowlers with NaN values
for player_name in missing_players_list:
    if player_name not in bowlers_players:
        new_row = {col: pd.NA for col in combined_bowlers.columns}
        new_row['Player'] = player_name
        combined_bowlers = pd.concat([combined_bowlers, pd.DataFrame([new_row])], ignore_index=True)

# Save updated dataframes
combined_batters.to_csv('/home/surya/AgentXI/data/new-combined-batters.csv', index=False)
combined_bowlers.to_csv('/home/surya/AgentXI/data/new-combined-bowlers.csv', index=False)

print(f"Added {len(missing_players_list)} missing players to combined data")

Added 69 missing players to combined data


In [ ]:
/home/surya/AgentXI/data/squads.json 
/home/surya/AgentXI/data/new-combined-batters.csv
/home/surya/AgentXI/data/new-combined-bowlers.csv
